In [ ]:
# imports
from dotenv import load_dotenv
from openai import OpenAI
import json
import os
import requests
from pypdf import PdfReader
import gradio as gr

In [ ]:
# The usual start
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
# Getting the Telegram bot token and chat ID from environment variables
# You can also replace these with your actual values directly

TELEGRAM_BOT_TOKEN = os.getenv("TELEGRAM_BOT_TOKEN", "your_bot_token_here")
TELEGRAM_CHAT_ID = os.getenv("TELEGRAM_CHAT_ID", "your_chat_id_here")

In [ ]:
def send_telegram_message(text, parse_mode=None):
    """Send a message to Telegram and return a normalized status payload."""
    if not TELEGRAM_BOT_TOKEN or TELEGRAM_BOT_TOKEN == "your_bot_token_here":
        return {"status": "error", "message": "TELEGRAM_BOT_TOKEN is not configured"}
    if not TELEGRAM_CHAT_ID or TELEGRAM_CHAT_ID == "your_chat_id_here":
        return {"status": "error", "message": "TELEGRAM_CHAT_ID is not configured"}

    url = f"https://api.telegram.org/bot{TELEGRAM_BOT_TOKEN}/sendMessage"
    payload = {"chat_id": TELEGRAM_CHAT_ID, "text": text}
    if parse_mode:
        payload["parse_mode"] = parse_mode

    try:
        response = requests.post(url, data=payload, timeout=10)
        response.raise_for_status()
        return {"status": "success", "message": text}
    except requests.RequestException as exc:
        return {"status": "error", "message": str(exc)}

In [ ]:
def record_user_details(email, name="Name not provided", notes="not provided", interest_level="medium"):
    """Record contact details from a user who may want follow-up."""
    email = (email or "").strip().lower()
    name = (name or "Name not provided").strip()
    notes = (notes or "not provided").strip()
    interest_level = (interest_level or "medium").strip().lower()

    if "@" not in email:
        return {"recorded": "error", "reason": "invalid_email"}

    text = (
        "[LEAD]\n"
        f"Name: {name}\n"
        f"Email: {email}\n"
        f"Interest level: {interest_level}\n"
        f"Notes: {notes}"
    )
    send_telegram_message(text)
    return {"recorded": "ok", "email": email}

In [ ]:
def _chunk_text(text, chunk_size=800, overlap=120):
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end].strip()
        if chunk:
            chunks.append(chunk)
        start = end - overlap
    return chunks


def _build_kb_chunks():
    """Build KB chunks from summary + LinkedIn for RAG."""
    summary_text = globals().get("summary", "") or ""
    linkedin_text = globals().get("linkedin", "") or ""
    kb_text = "\n\n".join([summary_text, linkedin_text]).strip()
    return _chunk_text(kb_text)


def _cosine_sim(a, b):
    dot = sum(x * y for x, y in zip(a, b))
    na = sum(x * x for x in a) ** 0.5
    nb = sum(y * y for y in b) ** 0.5
    return dot / (na * nb + 1e-8)


def _get_kb_index():
    """Compute and cache in-memory embeddings for KB chunks (no external DB)."""
    cache = globals().get("_kb_cache")
    if cache:
        return cache

    chunks = _build_kb_chunks()
    if not chunks:
        globals()["_kb_cache"] = {"chunks": [], "embeddings": []}
        return globals()["_kb_cache"]

    response = openai.embeddings.create(
        model="text-embedding-3-small",
        input=chunks,
    )
    embeddings = [item.embedding for item in response.data]
    globals()["_kb_cache"] = {"chunks": chunks, "embeddings": embeddings}
    return globals()["_kb_cache"]


def search_resume_context(query, top_k=3):
    """Return top matching resume snippets using in-memory embeddings."""
    query = (query or "").strip()
    if not query:
        return {"query": query, "matches": []}

    kb = _get_kb_index()
    if not kb["chunks"]:
        return {"query": query, "matches": []}

    q_resp = openai.embeddings.create(
        model="text-embedding-3-small",
        input=[query],
    )
    q_emb = q_resp.data[0].embedding

    scored = []
    for chunk, emb in zip(kb["chunks"], kb["embeddings"]):
        scored.append((_cosine_sim(q_emb, emb), chunk))

    scored.sort(key=lambda x: x[0], reverse=True)
    matches = [text for _, text in scored[: max(1, int(top_k))]]
    return {"query": query, "matches": matches}


def _looks_off_topic(question):
    """Heuristic for personal-preference or unrelated questions."""
    q = (question or "").lower()
    preference_keywords = [
        "favorite color",
        "favorite",
        "favourite",
        "music",
        "movie",
        "food",
        "hobby",
        "hobbies",
        "sports team",
        "pet",
        "personal preference",
    ]
    return any(k in q for k in preference_keywords)


def _should_log_unknown(question, assistant_reply, matches):
    """Log if we have no grounding and the reply indicates non-answer/off-topic."""
    reply = (assistant_reply or "").lower()
    if matches:
        return False
    if _looks_off_topic(question):
        return True
    non_answer_signals = [
        "i don't have personal preferences",
        "i do not have personal preferences",
        "as a professional representation",
        "i don't have a personal preference",
        "i do not have a personal preference",
        "i don't know",
        "i do not know",
    ]
    return any(s in reply for s in non_answer_signals)


def record_unknown_question(question, reason="insufficient_context", attempted_answer=""):
    """Record unanswered questions so they can be reviewed later."""
    text = (
        "[UNKNOWN QUESTION]\n"
        f"Question: {question}\n"
        f"Reason: {reason}\n"
        f"Attempted answer: {attempted_answer if attempted_answer else 'none'}"
    )
    send_telegram_message(text)
    return {"recorded": "ok", "reason": reason}


def record_hard_question(question, user_name="", follow_up_needed=True):
    """Escalate difficult questions for manual review via Telegram."""
    text = (
        "[HARD QUESTION ESCALATION]\n"
        f"User: {user_name if user_name else 'unknown'}\n"
        f"Question: {question}\n"
        f"Follow-up needed: {str(bool(follow_up_needed))}"
    )
    send_telegram_message(text)
    return {"recorded": "ok", "follow_up_needed": bool(follow_up_needed)}

In [ ]:
record_user_details_json = {
    "name": "record_user_details",
    "description": "Use this tool to record that a user is interested in being in touch and provided an email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {
                "type": "string",
                "description": "The email address of this user",
            },
            "name": {
                "type": "string",
                "description": "The user's name, if they provided it",
            },
            "notes": {
                "type": "string",
                "description": "Any additional information about the conversation that's worth recording to give context",
            },
            "interest_level": {
                "type": "string",
                "description": "Lead quality estimate: low, medium, or high",
                "enum": ["low", "medium", "high"],
            },
        },
        "required": ["email"],
        "additionalProperties": False,
    },
}

In [ ]:
record_unknown_question_json = {
    "name": "record_unknown_question",
    "description": "Use this tool to record questions that cannot be answered with confidence from the profile context.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The question that couldn't be answered",
            },
            "reason": {
                "type": "string",
                "description": "Why the answer was not possible",
            },
            "attempted_answer": {
                "type": "string",
                "description": "Optional partial answer attempted before escalation",
            },
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

search_resume_context_json = {
    "name": "search_resume_context",
    "description": "Search summary and LinkedIn profile text for relevant snippets before answering hard questions.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "The user's question or search query",
            },
            "top_k": {
                "type": "integer",
                "description": "How many snippets to return",
                "minimum": 1,
                "maximum": 10,
            },
        },
        "required": ["query"],
        "additionalProperties": False,
    },
}

record_hard_question_json = {
    "name": "record_hard_question",
    "description": "Escalate high-difficulty questions for manual follow-up via Telegram.",
    "parameters": {
        "type": "object",
        "properties": {
            "question": {
                "type": "string",
                "description": "The difficult question asked by the user",
            },
            "user_name": {
                "type": "string",
                "description": "Optional user name if known",
            },
            "follow_up_needed": {
                "type": "boolean",
                "description": "Whether this requires manual follow-up",
            },
        },
        "required": ["question"],
        "additionalProperties": False,
    },
}

In [ ]:
tools = [
    {"type": "function", "function": record_user_details_json},
    {"type": "function", "function": record_unknown_question_json},
    {"type": "function", "function": search_resume_context_json},
    {"type": "function", "function": record_hard_question_json},
]

In [ ]:
# This function can take a list of tool calls, and run them. This is the IF statement!!


def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)

        # THE BIG IF STATEMENT!!!

        if tool_name == "record_user_details":
            result = record_user_details(**arguments)
        elif tool_name == "record_unknown_question":
            result = record_unknown_question(**arguments)

        results.append(
            {
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id,
            }
        )
    return results

In [ ]:
# This is a more elegant way that avoids the IF statement.


def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        print(f"Tool called: {tool_name}", flush=True)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append(
            {
                "role": "tool",
                "content": json.dumps(result),
                "tool_call_id": tool_call.id,
            }
        )
    return results

In [ ]:
reader = PdfReader("Profile.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

with open("summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

name = "Derya Umut Kulali"

In [ ]:
system_prompt = f"You are acting as {name}. You are answering questions on {name}'s website, \
particularly questions related to {name}'s career, background, skills and experience. \
Your responsibility is to represent {name} for interactions on the website as faithfully as possible. \
You are given a summary of {name}'s background and LinkedIn profile which you can use to answer questions. \
Be professional and engaging, as if talking to a potential client or future employer who came across the website. \
Before saying you don't know, call search_resume_context to look for supporting context. \
If you still cannot answer confidently, call record_unknown_question to log it. \
If the question is clearly difficult, high-stakes, or needs manual follow-up, call record_hard_question as well. \
For personal-preference or unrelated questions (e.g., favorite color), politely decline and ALWAYS call record_unknown_question. \
If the user is engaging in discussion, try to steer them towards getting in touch via email; ask for their email and record it using your record_user_details tool. "

system_prompt += f"\n\n## Summary:\n{summary}\n\n## LinkedIn Profile:\n{linkedin}\n\n"
system_prompt += f"With this context, please chat with the user, always staying in character as {name}."

In [ ]:
def chat(message, history):
    # RAG context (in-memory embeddings, no external DB)
    rag = search_resume_context(message, top_k=3)
    rag_matches = rag.get("matches", []) if isinstance(rag, dict) else []
    rag_context = "\n\n".join(rag_matches)

    system = system_prompt
    if rag_context:
        system = system_prompt + f"\n\n## Retrieved context (for grounding):\n{rag_context}"

    messages = (
        [{"role": "system", "content": system}]
        + history
        + [{"role": "user", "content": message}]
    )

    # Avoid infinite loops if the model keeps requesting tools.
    max_tool_rounds = 6

    for _ in range(max_tool_rounds):
        # This is the call to the LLM - see that we pass in the tools json
        response = openai.chat.completions.create(
            model="gpt-4o-mini",
            messages=messages,
            tools=tools,
            tool_choice="auto",
        )

        finish_reason = response.choices[0].finish_reason

        # If the LLM wants to call a tool, we do that!
        if finish_reason == "tool_calls":
            assistant_message = response.choices[0].message
            tool_calls = assistant_message.tool_calls
            results = handle_tool_calls(tool_calls)
            messages.append(assistant_message)
            messages.extend(results)
            continue

        # Normal assistant response path
        assistant_reply = response.choices[0].message.content

        # Auto-log off-topic or unanswerable questions that slipped through
        matches = rag_matches
        if _should_log_unknown(message, assistant_reply, matches):
            record_unknown_question(
                question=message,
                reason="off_topic_or_unanswerable",
                attempted_answer=assistant_reply,
            )

        return assistant_reply

    # Fallback if tool loop did not converge
    return "I need a quick manual follow-up for this request. Could you share your email so I can get back to you with a precise answer?"

In [ ]:
gr.ChatInterface(chat, type="messages").launch(inbrowser=True)